In [0]:
spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "json")\
    .option("cloudFiles.schemaLocation", "/FileStore/tables/schema")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .load("/FileStore/tables/")\
    .writeStream.format("delta").option("checkpointLocation", "/FileStore/tables/checkpoint")\
    .start("/FileStore/tables/delta")

In [0]:
d = "/Volumes/data/raw/sales/tmp_stream_data"
checkpoints = "/Volumes/data/raw/sales/checkpoints/orders_stream"

spark.createDataFrame([("Hello",), ("World",)]) \
    .write.mode("overwrite").format("text").save(d)

q = spark.readStream.format("text") \
    .load(d) \
    .writeStream.format("console") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", checkpoints) \
    .start()

q.awaitTermination()

In [0]:
dbutils.fs.ls("/Volumes/data/raw/sales/partitioned_data/retail_db/orders/")
# dbutils.fs.ls("/Volumes/data/raw/sales/checkpoints/orders_stream")

In [0]:
dbutils.fs.rm("dbfs:/Volumes/data/raw/sales/checkpoints", recurse=True)

In [0]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("dbfs:/Volumes/data/raw/sales/partitioned_data/retail_db/orders/")

In [0]:
# df.limit(10).display()
df.printSchema()

In [0]:
df.select("order_date").distinct().display()

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales")

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales/retail_db/")

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales/retail_db/orders")

In [0]:
original_data_df = spark.read \
    .format("csv") \
    .load("dbfs:/Volumes/data/raw/sales/retail_db/orders")

In [0]:
# original_data_df.limit(5).display()
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

# original_data_df.select(col("_c1").cast(DateType())).distinct().display()
original_data_df.filter(col("_c1").cast(DateType()) == "2013-08-03").display()


In [0]:
original_data_df.printSchema()

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales")

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales/read_input/orders/")

In [0]:
read_input_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("dbfs:/Volumes/data/raw/sales/read_input/orders/")
read_input_df.orderBy("order_id").limit(5).display()

In [0]:
read_input_df.printSchema()

In [0]:
from pyspark.sql.functions import col
read_input_df.select(col("order_id").cast("integer")).orderBy("order_id").limit(5).display()

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
read_input_df.select(col("order_id").cast(IntegerType())).orderBy("order_id").limit(5).display()

In [0]:
%sql
show tables in sales.bronze;

In [0]:
%sql
select * from sales.bronze.customers_raw;

In [0]:
%sql
describe table extended sales.bronze.customers_raw;

In [0]:
display(_sqldf)

In [0]:
_sqldf.select("customer_fname", "customer_lname").display()

In [0]:
customers_raw_df = spark.read.table("sales.bronze.customers_raw")

In [0]:
customers_raw_df.count()

In [0]:
%sql
-- CREATE EXTERNAL VOLUME sales.bronze.vol_sales
-- LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/sales/vol_sales';

In [0]:
%sql
show volumes in sales.bronze;
-- describe volume data.raw.sales;
-- drop volume sales.bronze.vol_sales;

In [0]:
# dbutils.fs.mkdirs("s3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/sales/vol_sales")
# dbutils.fs.rm("s3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/sales/volumes")

In [0]:
# dbutils.fs.ls("s3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/sales/")
dbutils.fs.ls("dbfs:/Volumes/sales/bronze/vol_sales/checkpoints")
dbutils.fs.ls("dbfs:/Volumes/sales/bronze/vol_sales/schemas")

In [0]:
# create check points location and chema location
# dbutils.fs.mkdirs("s3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/sales/vol_sales/checkpoints/customers")
# dbutils.fs.mkdirs("dbfs:/Volumes/sales/bronze/vol_sales/checkpoints/orders")

# dbutils.fs.mkdirs("s3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/sales/vol_sales/schemas/customers")
# dbutils.fs.mkdirs("dbfs:/Volumes/sales/bronze/vol_sales/schemas/orders")

In [0]:
dbutils.notebook.exit("Restarting")

In [0]:
checkpoints = "dbfs:/Volumes/sales/bronze/vol_sales/checkpoints/orders"
schemas = "dbfs:/Volumes/sales/bronze/vol_sales/schemas/orders"
raw_data_orders = "dbfs:/Volumes/data/raw/sales/read_input/orders"
# auto loader read data from volume dbfs:/Volumes/data/raw/orders
df_orders_2 = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schemas)
      .load(raw_data_orders))


In [0]:
df_orders_2.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", checkpoints) \
    .toTable("sales.bronze.orders_raw")
            
            

In [0]:
spark.read.table("sales.bronze.orders_raw").printSchema()

In [0]:
df_orders_2.printSchema()

In [0]:
display(df_orders_2)

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales/read_input/orders")

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales/partitioned_data/retail_db/orders")

In [0]:
dbutils.fs.cp("/Volumes/data/raw/sales/partitioned_data/retail_db/orders/order_date=2013-08-08","/Volumes/data/raw/sales/read_input/orders/order_date=2013-08-08", recurse=True) 

In [0]:
from pyspark.sql.functions import try_to_date
help(try_to_date)